In [2]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [3]:
text_model = load_model("../models/text_lstm_genre_model.keras")
midi_model = load_model("../models/midi_lstm_model.keras")

print("Models loaded successfully!")


Models loaded successfully!


In [4]:
with open("../models/text_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("../models/genre_encoder.pkl", "rb") as f:
    genre_encoder = pickle.load(f)

with open("../models/int_to_note.pkl", "rb") as f:
    int_to_note = pickle.load(f)

with open("../models/note_to_int.pkl", "rb") as f:
    note_to_int = pickle.load(f)


In [5]:
X_midi = np.load("../models/X_midi.npy")


In [30]:
def predict_genre(text):
    text = text.lower()

    if "rock" in text:
        return "Rock"
    if "jazz" in text:
        return "Jazz"
    if "classical" in text:
        return "Classical"
    if "edm" in text:
        return "EDM"
    if "hip hop" in text:
        return "Hip-Hop"
    if "ambient" in text:
        return "Ambient"

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=50)
    pred = text_model.predict(padded)

    return genre_encoder.inverse_transform([np.argmax(pred)])[0]

In [7]:
def get_seed_by_genre(genre):
    if genre == "Classical":
        start = 0
    elif genre == "Jazz":
        start = len(X_midi)//6
    elif genre == "Rock":
        start = 2*len(X_midi)//6
    elif genre == "EDM":
        start = 3*len(X_midi)//6
    elif genre == "Hip-Hop":
        start = 4*len(X_midi)//6
    else:  # Ambient
        start = 5*len(X_midi)//6

    idx = np.random.randint(start, start + len(X_midi)//6)

    pattern = X_midi[idx]
    return pattern.reshape(1, len(pattern), 1)


In [8]:
def sample_with_temperature(preds, temperature=0.8):
    preds = preds.astype("float64")
    preds = np.log(preds + 1e-9) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)


In [9]:
def generate_music(seed_pattern, length=300):
    pattern = seed_pattern.copy()
    generated_notes = []

    for _ in range(length):

        # Predict next-note probabilities
        prediction = midi_model.predict(pattern, verbose=0)[0]

        # Sample instead of argmax (THIS creates variation)
        index = sample_with_temperature(prediction, temperature=0.8)

        result = int_to_note[index]
        generated_notes.append(result)

        # Slide window forward
        pattern = np.append(pattern[:, 1:, :], [[[index]]], axis=1)

    return generated_notes


In [10]:
print(genre_encoder.classes_)

['Ambient' 'Classical' 'EDM' 'Hip-Hop' 'Jazz' 'Rock']


In [11]:
print(tokenizer.word_index.get("rock"))

30


In [12]:
from music21 import stream, note, chord, instrument, tempo

def save_midi(generated_notes, genre, filename="generated_music.mid"):
    midi_stream = stream.Stream()

    # 🎼 Assign instrument based on genre
    if genre == "Rock":
        midi_stream.append(instrument.ElectricGuitar())
        midi_stream.append(tempo.MetronomeMark(number=140))

    elif genre == "Jazz":
        midi_stream.append(instrument.Saxophone())
        midi_stream.append(tempo.MetronomeMark(number=120))

    elif genre == "Classical":
        midi_stream.append(instrument.Piano())
        midi_stream.append(tempo.MetronomeMark(number=90))

    elif genre == "EDM":
        midi_stream.append(instrument.ElectricBass())
        midi_stream.append(tempo.MetronomeMark(number=128))

    elif genre == "Hip-Hop":
        midi_stream.append(instrument.ElectricBass())
        midi_stream.append(tempo.MetronomeMark(number=85))

    else:  # Ambient
        midi_stream.append(instrument.StringEnsemble())
        midi_stream.append(tempo.MetronomeMark(number=60))

    # 🎵 Convert generated tokens → actual notes
    for pattern in generated_notes:

        pattern = str(pattern)

        # Case 1: chord like "0.4.7"
        if '.' in pattern:
            notes = pattern.split('.')
            chord_notes = []

            for n in notes:
                try:
                    chord_notes.append(note.Note(int(n) + 60))  # shift to audible range
                except:
                    chord_notes.append(note.Note(n))

            midi_stream.append(chord.Chord(chord_notes))

        # Case 2: numeric pitch like "5"
        elif pattern.isdigit():
            midi_number = int(pattern) + 60
            midi_stream.append(note.Note(midi_number))

        # Case 3: named pitch like "C4"
        else:
            try:
                midi_stream.append(note.Note(pattern))
            except:
                midi_stream.append(note.Note("C4"))  # fallback safety

    # 💾 Save MIDI file
    midi_stream.write('midi', fp=filename)
    print(f"{genre} MIDI saved as {filename}")


In [13]:
print("Genres:", genre_encoder.classes_)


Genres: ['Ambient' 'Classical' 'EDM' 'Hip-Hop' 'Jazz' 'Rock']


In [14]:
predict_genre("soft violin classical music")
predict_genre("heavy electric guitar rock")
predict_genre("smooth saxophone jazz")
predict_genre("upbeat electronic dance track")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
Prediction vector: [[9.1072679e-01 8.9153741e-04 6.2019692e-04 1.0585903e-10 6.5985374e-02
  2.1776080e-02]]
Predicted index: 0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Prediction vector: [[1.14399445e-04 1.50217620e-05 1.77995637e-02 7.72656852e-12
  2.31177983e-05 9.82047915e-01]]
Predicted index: 5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Prediction vector: [[9.1667793e-04 9.9685919e-01 8.0113859e-05 6.6104359e-13 6.0574181e-04
  1.5383093e-03]]
Predicted index: 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction vector: [[2.49503911e-01 1.15334224e-04 7.28992673e-05 3.54244249e-13
  7.48900473e-01 1.40731537e-03]]
Predicted index: 4


'Jazz'

In [28]:
seq = tokenizer.texts_to_sequences(["Generate a fast rock music track"])
print(seq)

[[7, 1, 5, 30, 8]]


In [27]:
print(text_model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (32, 50, 64)           │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (32, 64)               │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (32, 64)               │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (32, 64)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (32, 6)                │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 973,652 (3.71 MB)

 Trainable params: 324,550 (1.24 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 649,102 (2.48 MB)

None


In [29]:
test_inputs = [
    "rock guitar music",
    "jazz saxophone melody",
    "classical piano composition",
    "edm electronic dance beat",
    "hip hop rap beat",
    "ambient relaxing soundscape"
]

for t in test_inputs:
    print(t, "→", predict_genre(t))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Prediction vector: [[2.7562636e-03 2.4911040e-04 1.1507939e-03 2.7029599e-11 3.4112895e-03
  9.9243248e-01]]
Predicted index: 5
rock guitar music → Rock
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
Prediction vector: [[9.1667747e-04 9.9685919e-01 8.0113859e-05 6.6104359e-13 6.0574239e-04
  1.5383093e-03]]
Predicted index: 1
jazz saxophone melody → Classical
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Prediction vector: [[4.2010576e-01 9.4744936e-03 1.2354964e-02 5.9798007e-11 2.7766448e-01
  2.8040025e-01]]
Predicted index: 0
classical piano composition → Ambient
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Prediction vector: [[2.6970629e-03 6.2069617e-04 1.7216121e-01 1.9194446e-09 1.0797660e-02
  8.1372333e-01]]
Predicted index: 5
edm electronic dance beat → Rock
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Prediction vector: [[2.0235304e-04 3.2027421e-04 1.2073600e-01 7.2124117e-06 4.9104523e-02
  8.2962966e-01]]
Predicted index: 5
hip hop rap beat → Rock
1/1 ━━━━━━━━━━━━

In [36]:
user_text = input("Describe the music you want: ")

genre = predict_genre(user_text)
print("Predicted Genre:", genre)

seed = get_seed_by_genre(genre)

generated_notes = generate_music(seed, length=400)

filename = f"generated_{genre}.mid"
save_midi(generated_notes, genre, filename)

print("Music generated!")


Predicted Genre: Rock


KeyboardInterrupt: 